In [76]:
from pathlib import Path
import pandas as pd
from rdkit import Chem
from rdkit.Chem import PandasTools, Descriptors, rdchem, rdMolDescriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.model_selection import cross_validate
from xgboost import XGBRegressor

def mol_to_feat(mol):
    #找我认为的重要键
    NO_single = NO_double = 0
    NN_single = NN_double = 0
    sum = 0
    for i in mol.GetBonds():
        pre = i.GetBeginAtom().GetSymbol()
        nxt = i.GetEndAtom().GetSymbol()
        link = i.GetBondType()
        if pre == "O":
            pre, nxt = nxt, pre
        if pre == "N" and nxt == "O":
            if link == rdchem.BondType.SINGLE:
                NO_single += 1
            else:
                NO_double += 1
        elif pre == "N" and nxt == "N":
            if link == rdchem.BondType.SINGLE:
                NN_single += 1
            else:
                NN_double += 1
        sum += 1

    #算各种原子的个数
    mqn = rdMolDescriptors.MQNs_(mol)
    N = mqn[7] + mqn[8]
    O = mqn[9] + mqn[10]
    C = mqn[0]
    mol = Chem.AddHs(mol)
    H = mol.GetNumAtoms() - mol.GetNumHeavyAtoms()
    
    feat = [
        (C * 2 + H / 2 - O) * 16 / Descriptors.MolWt(mol),
        Descriptors.fr_Ar_N(mol) * 14 / Descriptors.MolWt(mol),
        NO_double / sum, NN_single, NN_double
    ]

    desc_names = ['EState_VSA8', 'VSA_EState3', 'PEOE_VSA1', 'PEOE_VSA7', "SlogP_VSA1", "SlogP_VSA4"]
    feat = feat + list(MoleculeDescriptors.MolecularDescriptorCalculator(desc_names).CalcDescriptors(mol))

    return feat

file = Path.cwd().parent / "data" / "test.xlsx"
df = pd.read_excel(file)
PandasTools.AddMoleculeColumnToFrame(df, "SMILES", "ROMol", False)
x_all = np.stack(StandardScaler().fit_transform(df["ROMol"].apply(mol_to_feat).tolist()), axis = 0)
y_all = np.array(df["Q(cal/g)"], dtype = np.float64)

clf = XGBRegressor(
    n_estimators = 2000,
    learning_rate = 0.05,
    max_depth = 6,
    reg_lambda = 1.0,
    n_jobs = -1,
    random_state = 323922
)

scores = cross_validate(clf, x_all, y_all, cv = 10, scoring = ["neg_mean_squared_error", "neg_mean_absolute_error", "r2"], n_jobs = -1)
rmse = ((-scores["test_neg_mean_squared_error"]) ** 0.5).mean()
mae = np.abs(scores["test_neg_mean_absolute_error"]).mean()
r2 = scores["test_r2"]
print(rmse, mae, r2, r2.mean(), sep = '\n')


162.08788162600845
130.3386263271928
[0.82762015 0.92307916 0.893024   0.69295906 0.68956469 0.74916059
 0.88955384 0.89173915 0.57434853 0.86016544]
0.7991214602653438
